[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_07/notebook_7_5_forecasting.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 7.5: Time Series Forecasting for Clinical Applications

## Learning Objectives

By the end of this notebook, you will understand:

1. **Vital sign forecasting**: Predicting future values of physiological signals (heart rate, blood pressure)
2. **Multi-step prediction**: Forecasting multiple time steps ahead with horizon uncertainty
3. **Uncertainty quantification**: Confidence intervals and prediction intervals for clinical decision-making
4. **Model comparison**: Classical methods (ARIMA, Exponential Smoothing) vs deep learning (LSTM)

## Clinical Context

**Connection to Journey 7.3**: Sepsis Prediction (forecasting vital sign deterioration 6 hours ahead)

**Why forecasting matters in healthcare**:

- **Early warning**: Predict patient deterioration before critical events (septic shock, cardiac arrest)
- **Resource allocation**: Anticipate ICU bed needs, staffing requirements
- **Treatment planning**: Forecast blood glucose to optimize insulin dosing
- **Epidemic modeling**: Predict disease spread (COVID-19, flu outbreaks)

**Real-world scenario**: In the ICU, continuously monitored heart rate can predict impending septic shock 4-6 hours before clinical deterioration. Early intervention (antibiotics, fluids) dramatically improves survival.

**Key challenges**:
- **Noise**: Physiological signals are noisy (artifacts, measurement errors)
- **Non-stationarity**: Patient condition changes over time (medications, disease progression)
- **Missing data**: Sensor disconnections, gaps in monitoring
- **Uncertainty**: Predictions must include confidence intervals for clinical decision-making

---

## Setup

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

# Statistical forecasting
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller, acf, pacf

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_style('whitegrid')

print("✓ Libraries imported successfully!")
print("\n📦 Key libraries:")
print("  - statsmodels: ARIMA, Exponential Smoothing")
print("  - NumPy/Pandas: Data manipulation")
print("  - Matplotlib: Visualization")

## 1. Generate Synthetic Vital Sign Time Series

We'll simulate heart rate data for two patient scenarios:
1. **Stable patient**: Normal variation, slight trend
2. **Deteriorating patient**: Heart rate increases (early sepsis), simulating the need for forecasting

**Heart rate characteristics**:
- **Normal range**: 60-100 BPM (beats per minute)
- **Sampling**: Every 5 minutes (common for ICU monitoring)
- **Trend**: Deteriorating patient shows gradual increase
- **Noise**: Gaussian noise + occasional artifacts

In [ ]:
def generate_vital_sign_timeseries(n_hours=24, sampling_interval_min=5, patient_type='stable'):
    """
    Generate synthetic heart rate time series.

    Args:
        n_hours: Duration in hours
        sampling_interval_min: Sampling interval in minutes
        patient_type: 'stable' or 'deteriorating'

    Returns:
        df: DataFrame with timestamp and heart_rate
    """
    # Number of samples
    n_samples = int(n_hours * 60 / sampling_interval_min)

    # Time index
    time = np.arange(n_samples) * sampling_interval_min / 60  # hours

    # Base heart rate
    if patient_type == 'stable':
        # Stable patient: mean ~75 BPM, slight circadian variation
        baseline = 75 + 5 * np.sin(2 * np.pi * time / 24)  # Circadian rhythm
        trend = 0
    elif patient_type == 'deteriorating':
        # Deteriorating patient: gradual increase (sepsis onset)
        baseline = 70
        trend = 3 * time  # +3 BPM per hour

    # Noise components
    # 1. Gaussian noise (measurement + physiological variation)
    noise = np.random.normal(0, 3, n_samples)

    # 2. Autoregressive component (short-term correlation)
    ar_noise = np.zeros(n_samples)
    ar_noise[0] = noise[0]
    for i in range(1, n_samples):
        ar_noise[i] = 0.7 * ar_noise[i-1] + 0.3 * noise[i]

    # 3. Occasional artifacts (motion, sensor issues)
    artifact_prob = 0.02
    artifacts = np.random.rand(n_samples) < artifact_prob
    artifact_values = np.random.uniform(-15, 15, n_samples) * artifacts

    # Combine
    heart_rate = baseline + trend + ar_noise + artifact_values
    heart_rate = np.clip(heart_rate, 40, 180)  # Physiological limits

    # Create DataFrame
    df = pd.DataFrame({
        'time_hours': time,
        'heart_rate': heart_rate
    })

    return df


# Generate data
n_hours = 48  # 2 days of monitoring
sampling_interval = 5  # minutes

df_stable = generate_vital_sign_timeseries(n_hours=n_hours, sampling_interval_min=sampling_interval,
                                            patient_type='stable')
df_deteriorating = generate_vital_sign_timeseries(n_hours=n_hours, sampling_interval_min=sampling_interval,
                                                   patient_type='deteriorating')

print(f"✓ Generated {n_hours}-hour vital sign time series")
print(f"  - Sampling interval: {sampling_interval} minutes")
print(f"  - Total samples: {len(df_stable)}")
print(f"  - Stable patient HR: {df_stable['heart_rate'].mean():.1f} ± {df_stable['heart_rate'].std():.1f} BPM")
print(f"  - Deteriorating patient HR: {df_deteriorating['heart_rate'].mean():.1f} ± {df_deteriorating['heart_rate'].std():.1f} BPM")

In [ ]:
# Visualize time series
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Stable patient
axes[0].plot(df_stable['time_hours'], df_stable['heart_rate'],
             linewidth=1, color='blue', alpha=0.7, label='Heart Rate')
axes[0].axhline(75, color='green', linestyle='--', alpha=0.5, label='Normal mean (75 BPM)')
axes[0].fill_between(df_stable['time_hours'], 60, 100, alpha=0.1, color='green', label='Normal range')
axes[0].set_ylabel('Heart Rate (BPM)', fontsize=11)
axes[0].set_title('Stable Patient: Heart Rate (48 hours)', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Deteriorating patient
axes[1].plot(df_deteriorating['time_hours'], df_deteriorating['heart_rate'],
             linewidth=1, color='red', alpha=0.7, label='Heart Rate')
axes[1].axhline(75, color='green', linestyle='--', alpha=0.5, label='Normal mean (75 BPM)')
axes[1].fill_between(df_deteriorating['time_hours'], 60, 100, alpha=0.1, color='green', label='Normal range')
axes[1].axvspan(36, 48, alpha=0.2, color='orange', label='Forecast horizon (12h)')
axes[1].set_xlabel('Time (hours)', fontsize=11)
axes[1].set_ylabel('Heart Rate (BPM)', fontsize=11)
axes[1].set_title('Deteriorating Patient: Heart Rate (48 hours, trending upward)',
                   fontsize=12, fontweight='bold')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 Key Observations:")
print("  - Stable patient: Circadian rhythm, minor fluctuations")
print("  - Deteriorating patient: Gradual upward trend (tachycardia onset)")
print("  - Goal: Forecast next 12 hours (orange shaded region) to predict deterioration")

## 2. Time Series Analysis: Stationarity and Autocorrelation

Before forecasting, we need to check two key properties:

### 2.1 Stationarity

A time series is **stationary** if its statistical properties (mean, variance) don't change over time.

**Why it matters**: Many forecasting models (ARIMA) assume stationarity.

**Augmented Dickey-Fuller (ADF) Test**:
- **Null hypothesis**: Time series is non-stationary (has a unit root)
- **p-value < 0.05**: Reject null → stationary
- **p-value ≥ 0.05**: Fail to reject → non-stationary (needs differencing)

### 2.2 Autocorrelation

**Autocorrelation Function (ACF)**: Correlation of the series with itself at different lags

$$
\rho(k) = \frac{\text{Cov}(y_t, y_{t-k})}{\text{Var}(y_t)}
$$

**Partial Autocorrelation Function (PACF)**: Correlation after removing effects of intermediate lags

**Use for ARIMA parameter selection**:
- **ACF decay + PACF cutoff**: AR(p) model
- **ACF cutoff + PACF decay**: MA(q) model
- **Both decay**: ARMA(p,q) model

In [ ]:
def test_stationarity(timeseries, title=''):
    """
    Perform Augmented Dickey-Fuller test for stationarity.

    Args:
        timeseries: Time series data
        title: Plot title
    """
    # Perform ADF test
    result = adfuller(timeseries, autolag='AIC')

    print(f"\n📊 Augmented Dickey-Fuller Test: {title}")
    print("="*60)
    print(f"ADF Statistic:      {result[0]:.4f}")
    print(f"p-value:            {result[1]:.4f}")
    print(f"Critical Values:")
    for key, value in result[4].items():
        print(f"  {key}: {value:.4f}")
    print("="*60)

    if result[1] < 0.05:
        print("✅ Result: Stationary (p < 0.05) - safe to use ARIMA")
    else:
        print("⚠️ Result: Non-stationary (p ≥ 0.05) - consider differencing")

    return result[1] < 0.05


# Test both series
is_stationary_stable = test_stationarity(df_stable['heart_rate'].values, 'Stable Patient')
is_stationary_deteriorating = test_stationarity(df_deteriorating['heart_rate'].values, 'Deteriorating Patient')

In [ ]:
# Plot ACF and PACF for deteriorating patient
fig, axes = plt.subplots(2, 2, figsize=(14, 7))

# Original series
axes[0, 0].plot(df_deteriorating['time_hours'], df_deteriorating['heart_rate'],
                linewidth=1, color='red')
axes[0, 0].set_xlabel('Time (hours)', fontsize=10)
axes[0, 0].set_ylabel('Heart Rate (BPM)', fontsize=10)
axes[0, 0].set_title('Original Series', fontsize=11, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Differenced series (remove trend)
diff_series = np.diff(df_deteriorating['heart_rate'].values)
axes[0, 1].plot(df_deteriorating['time_hours'][1:], diff_series,
                linewidth=1, color='blue')
axes[0, 1].set_xlabel('Time (hours)', fontsize=10)
axes[0, 1].set_ylabel('Δ Heart Rate (BPM)', fontsize=10)
axes[0, 1].set_title('Differenced Series (stationary)', fontsize=11, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# ACF
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
plot_acf(diff_series, lags=40, ax=axes[1, 0], alpha=0.05)
axes[1, 0].set_title('Autocorrelation Function (ACF)', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Lag', fontsize=10)

# PACF
plot_pacf(diff_series, lags=40, ax=axes[1, 1], alpha=0.05)
axes[1, 1].set_title('Partial Autocorrelation Function (PACF)', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Lag', fontsize=10)

plt.tight_layout()
plt.show()

print("\n🔍 Interpretation:")
print("  - Original series: Non-stationary (upward trend)")
print("  - Differenced series: Stationary (trend removed)")
print("  - ACF: Decay pattern suggests AR component")
print("  - PACF: Cutoff at lag ~2 suggests AR(2) model")
print("  ➡️ Recommendation: ARIMA(2, 1, 0) - AR(2) with 1st-order differencing")

## 3. Classical Forecasting: ARIMA

**ARIMA (AutoRegressive Integrated Moving Average)** is the gold standard for univariate time series forecasting.

### Model Structure: ARIMA(p, d, q)

**1. AR (AutoRegressive) - p**: Current value depends on past p values
$$
y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + \cdots + \phi_p y_{t-p} + \epsilon_t
$$

**2. I (Integrated) - d**: Number of differencing operations to achieve stationarity
$$
\Delta y_t = y_t - y_{t-1}
$$

**3. MA (Moving Average) - q**: Current value depends on past q forecast errors
$$
y_t = c + \epsilon_t + \theta_1 \epsilon_{t-1} + \theta_2 \epsilon_{t-2} + \cdots + \theta_q \epsilon_{t-q}
$$

### Parameter Selection

- **p**: PACF cutoff lag
- **d**: Number of differences needed for stationarity (usually 1)
- **q**: ACF cutoff lag

**For our deteriorating patient**: ARIMA(2, 1, 0) based on ACF/PACF analysis

In [ ]:
# Split data: Train on first 36 hours, forecast next 12 hours
train_hours = 36
forecast_hours = 12

# Convert to indices
train_size = int(train_hours * 60 / sampling_interval)
forecast_size = int(forecast_hours * 60 / sampling_interval)

# Split
train_data = df_deteriorating['heart_rate'].values[:train_size]
test_data = df_deteriorating['heart_rate'].values[train_size:train_size + forecast_size]

print(f"\n📊 Data Split:")
print(f"  - Training set: {len(train_data)} samples ({train_hours} hours)")
print(f"  - Test set: {len(test_data)} samples ({forecast_hours} hours)")
print(f"  - Total: {len(df_deteriorating)} samples ({n_hours} hours)")

In [ ]:
# Fit ARIMA model
print("\n🔄 Training ARIMA(2, 1, 0) model...")

# Order: (p, d, q) = (2, 1, 0)
model_arima = ARIMA(train_data, order=(2, 1, 0))
fitted_arima = model_arima.fit()

print("✓ ARIMA model trained")
print("\n📊 Model Summary:")
print(fitted_arima.summary())

In [ ]:
# Forecast next 12 hours
forecast_arima = fitted_arima.forecast(steps=forecast_size)

# Get confidence intervals
forecast_result = fitted_arima.get_forecast(steps=forecast_size)
forecast_ci = forecast_result.conf_int(alpha=0.05)  # 95% confidence interval

# Calculate metrics
mae_arima = mean_absolute_error(test_data, forecast_arima)
rmse_arima = np.sqrt(mean_squared_error(test_data, forecast_arima))
mape_arima = np.mean(np.abs((test_data - forecast_arima) / test_data)) * 100

print("\n🎯 ARIMA Forecast Performance:")
print("="*60)
print(f"MAE (Mean Absolute Error):        {mae_arima:.2f} BPM")
print(f"RMSE (Root Mean Squared Error):   {rmse_arima:.2f} BPM")
print(f"MAPE (Mean Absolute % Error):     {mape_arima:.2f}%")
print("="*60)

if mae_arima < 5:
    print("✅ Excellent forecast accuracy (MAE < 5 BPM)")
elif mae_arima < 10:
    print("⚠️ Good forecast accuracy (MAE < 10 BPM)")
else:
    print("❌ Poor forecast accuracy (MAE ≥ 10 BPM)")

In [ ]:
# Visualize ARIMA forecast
fig, ax = plt.subplots(figsize=(14, 6))

# Training data
time_train = df_deteriorating['time_hours'].values[:train_size]
ax.plot(time_train, train_data, linewidth=1.5, color='blue', label='Training Data (36h)', alpha=0.7)

# Test data (ground truth)
time_test = df_deteriorating['time_hours'].values[train_size:train_size + forecast_size]
ax.plot(time_test, test_data, linewidth=2, color='black', label='Ground Truth (12h)', marker='o', markersize=3)

# ARIMA forecast
ax.plot(time_test, forecast_arima, linewidth=2, color='red', label='ARIMA Forecast',
        linestyle='--', marker='x', markersize=4)

# Confidence interval
ax.fill_between(time_test, forecast_ci[:, 0], forecast_ci[:, 1],
                alpha=0.2, color='red', label='95% Confidence Interval')

# Forecast horizon
ax.axvline(train_hours, color='green', linestyle=':', linewidth=2, alpha=0.7, label='Forecast Start')
ax.axvspan(train_hours, train_hours + forecast_hours, alpha=0.1, color='orange')

ax.set_xlabel('Time (hours)', fontsize=11)
ax.set_ylabel('Heart Rate (BPM)', fontsize=11)
ax.set_title(f'ARIMA(2,1,0) Forecast: Heart Rate (MAE={mae_arima:.2f} BPM)',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 Key Observations:")
print("  - ARIMA captures upward trend successfully")
print("  - Confidence interval widens as forecast horizon increases (uncertainty growth)")
print("  - Clinically useful: predicts heart rate will exceed 120 BPM (tachycardia)")

## 4. Exponential Smoothing (Holt-Winters)

**Exponential Smoothing** is another classical approach, particularly effective for data with **trend** and **seasonality**.

### Triple Exponential Smoothing (Holt-Winters)

Three components:

1. **Level** $\ell_t$: Smoothed average
   $$\ell_t = \alpha (y_t - s_{t-m}) + (1 - \alpha)(\ell_{t-1} + b_{t-1})$$

2. **Trend** $b_t$: Rate of change
   $$b_t = \beta (\ell_t - \ell_{t-1}) + (1 - \beta) b_{t-1}$$

3. **Seasonality** $s_t$: Periodic pattern
   $$s_t = \gamma (y_t - \ell_t) + (1 - \gamma) s_{t-m}$$

**Forecast**:
$$
\hat{y}_{t+h} = \ell_t + h b_t + s_{t+h-m}
$$

**When to use**:
- ✅ Strong trend or seasonality (circadian rhythm in vital signs)
- ✅ Simpler than ARIMA (fewer parameters)
- ❌ Less flexible than ARIMA for complex autocorrelation

In [ ]:
# Fit Holt-Winters model
print("\n🔄 Training Holt-Winters (Triple Exponential Smoothing) model...")

# No seasonality for this short series (use 'add' trend only)
model_hw = ExponentialSmoothing(
    train_data,
    trend='add',          # Additive trend
    seasonal=None,        # No seasonality (too short)
    damped_trend=False    # No damping
)
fitted_hw = model_hw.fit()

print("✓ Holt-Winters model trained")

# Forecast
forecast_hw = fitted_hw.forecast(steps=forecast_size)

# Calculate metrics
mae_hw = mean_absolute_error(test_data, forecast_hw)
rmse_hw = np.sqrt(mean_squared_error(test_data, forecast_hw))
mape_hw = np.mean(np.abs((test_data - forecast_hw) / test_data)) * 100

print("\n🎯 Holt-Winters Forecast Performance:")
print("="*60)
print(f"MAE (Mean Absolute Error):        {mae_hw:.2f} BPM")
print(f"RMSE (Root Mean Squared Error):   {rmse_hw:.2f} BPM")
print(f"MAPE (Mean Absolute % Error):     {mape_hw:.2f}%")
print("="*60)

In [ ]:
# Compare ARIMA vs Holt-Winters
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for i, (forecast, mae, title, color) in enumerate([
    (forecast_arima, mae_arima, 'ARIMA(2,1,0)', 'red'),
    (forecast_hw, mae_hw, 'Holt-Winters', 'purple')
]):
    # Training data
    axes[i].plot(time_train, train_data, linewidth=1.5, color='blue',
                 label='Training Data', alpha=0.5)

    # Ground truth
    axes[i].plot(time_test, test_data, linewidth=2, color='black',
                 label='Ground Truth', marker='o', markersize=3)

    # Forecast
    axes[i].plot(time_test, forecast, linewidth=2, color=color,
                 label=f'{title} Forecast', linestyle='--', marker='x', markersize=4)

    axes[i].axvline(train_hours, color='green', linestyle=':', linewidth=2, alpha=0.5)
    axes[i].set_ylabel('Heart Rate (BPM)', fontsize=11)
    axes[i].set_title(f'{title}: MAE = {mae:.2f} BPM', fontsize=12, fontweight='bold')
    axes[i].legend(loc='upper left')
    axes[i].grid(True, alpha=0.3)

axes[1].set_xlabel('Time (hours)', fontsize=11)

plt.tight_layout()
plt.show()

print("\n📊 Model Comparison:")
print("="*70)
print(f"{'Model':<20} {'MAE':>10} {'RMSE':>10} {'MAPE':>10}")
print("="*70)
print(f"{'ARIMA(2,1,0)':<20} {mae_arima:>10.2f} {rmse_arima:>10.2f} {mape_arima:>10.2f}%")
print(f"{'Holt-Winters':<20} {mae_hw:>10.2f} {rmse_hw:>10.2f} {mape_hw:>10.2f}%")
print("="*70)

if mae_arima < mae_hw:
    print("\n✅ Winner: ARIMA (lower MAE)")
else:
    print("\n✅ Winner: Holt-Winters (lower MAE)")

## 5. Multi-Step Forecasting and Uncertainty

### Multi-Step Ahead Forecasting

Two strategies:

1. **Direct**: Train separate model for each horizon (h=1, h=2, ..., h=12)
2. **Recursive**: Forecast 1-step, feed prediction back as input, repeat

**Trade-offs**:
- Direct: More accurate short-term, requires more models
- Recursive: Simpler, but errors accumulate over time

### Uncertainty Quantification

**Why it matters**: In clinical settings, we need to know confidence in predictions:
- **High confidence**: Proceed with intervention
- **Low confidence**: Request additional monitoring, human review

**Prediction intervals**: Range where future value is likely to fall
- **95% interval**: True value falls within interval 95% of the time
- **Interval width**: Measure of uncertainty (wider = more uncertain)

**Formula for ARIMA prediction interval**:

$$
\hat{y}_{t+h} \pm z_{\alpha/2} \cdot \sigma \sqrt{\sum_{i=0}^{h-1} \psi_i^2}
$$

where $\psi_i$ are MA coefficients, $\sigma$ is residual std, $z_{\alpha/2}=1.96$ for 95% CI.

In [ ]:
# Analyze uncertainty growth with forecast horizon
horizons = [6, 12, 24, 48]  # 0.5h, 1h, 2h, 4h ahead
uncertainties = []

print("\n🔍 Uncertainty vs Forecast Horizon:")
print("="*60)
print(f"{'Horizon (hours)':<20} {'CI Width (BPM)':>20} {'Uncertainty':>15}")
print("="*60)

for h in horizons:
    h_steps = int(h * 60 / sampling_interval)
    forecast_result = fitted_arima.get_forecast(steps=h_steps)
    forecast_ci = forecast_result.conf_int(alpha=0.05)

    # ERROR WAS HERE: forecast_ci.iloc[-1, 1] - forecast_ci.iloc[-1, 0]
    # FIXED: Use numpy indexing
    ci_width = forecast_ci[-1, 1] - forecast_ci[-1, 0]

    uncertainties.append(ci_width)

    print(f"{h / 60:<20.1f} {ci_width:>20.2f} {'█' * int(ci_width / 2):>15}")

print("="*60)
print("\n🔍 Key Insight: Uncertainty grows with forecast horizon")
print("  - Short-term (0.5h): Narrow CI, high confidence")
print("  - Long-term (4h): Wide CI, lower confidence")
print("  - Clinical implication: Trust short-term forecasts more")

In [ ]:
# Visualize uncertainty growth
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot([h / 60 for h in horizons], uncertainties,
        marker='o', markersize=8, linewidth=2, color='steelblue')
ax.set_xlabel('Forecast Horizon (hours)', fontsize=11)
ax.set_ylabel('95% Confidence Interval Width (BPM)', fontsize=11)
ax.set_title('Uncertainty Growth with Forecast Horizon', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Clinical Decision-Making:")
print("  - If CI width < 10 BPM: High confidence → automate decision")
print("  - If CI width 10-20 BPM: Moderate confidence → flag for review")
print("  - If CI width > 20 BPM: Low confidence → require human oversight")

## 6. Comparison: Classical vs Deep Learning

### When to Use Each Approach

| Aspect | ARIMA / Exponential Smoothing | LSTM / Transformer |
|--------|-------------------------------|--------------------|
| **Data required** | 100-1,000 samples | 10,000-1M+ samples |
| **Training time** | Seconds-minutes | Hours-days (GPU) |
| **Interpretability** | High (coefficients) | Low (black box) |
| **Multivariate** | Limited (VARIMA) | Excellent (multi-input) |
| **Non-linearity** | Linear only | Handles complex patterns |
| **Uncertainty** | Built-in (CI) | Requires MC Dropout |
| **Deployment** | CPU, lightweight | GPU, heavyweight |
| **Best for** | Short-term, univariate | Long-term, multivariate |

### LSTM (Long Short-Term Memory)

**Architecture**:
- **Input gate**: What new information to add
- **Forget gate**: What information to discard
- **Output gate**: What information to output

**Advantages**:
- Captures long-range dependencies (e.g., circadian patterns)
- Handles multivariate inputs (HR + BP + SpO2)
- Non-linear patterns (e.g., sudden deterioration)

**Disadvantages**:
- Requires large datasets (>10,000 samples)
- Computationally expensive
- Harder to interpret

**Note**: LSTM implementation would require TensorFlow/PyTorch (not included in this notebook). For demonstration, we've shown classical approaches that work well with limited data.

## 7. Real-World Considerations

### 7.1 Clinical Deployment Challenges

**1. Missing data**:
- **Problem**: Sensor disconnections, patient transfers, gaps in monitoring
- **Solutions**: Forward-fill, interpolation, or model with missingness indicators

**2. Concept drift**:
- **Problem**: Patient condition changes (medications, interventions alter signal patterns)
- **Solutions**: Regular model retraining, adaptive learning, patient-specific models

**3. Alert fatigue**:
- **Problem**: Too many false alarms → clinicians ignore warnings
- **Solutions**: High confidence thresholds, ensemble models, human-in-the-loop

**4. Computational constraints**:
- **Problem**: Real-time forecasting at bedside (latency < 1 second)
- **Solutions**: Lightweight models (ARIMA), edge computing, model compression

### 7.2 Regulatory Requirements

**FDA/CE Mark**:
- Validate on diverse populations (age, sex, comorbidities)
- Demonstrate prospective performance (not just retrospective)
- Document uncertainty quantification (prediction intervals)
- Provide clinical action thresholds

**Example**: If forecasting sepsis risk 6 hours ahead:
- **Low risk** (<10%): Continue standard monitoring
- **Medium risk** (10-30%): Increase monitoring frequency
- **High risk** (>30%): Alert physician, initiate sepsis bundle

### 7.3 Performance Metrics

**Standard metrics**:
- **MAE**: Average absolute error (interpretable)
- **RMSE**: Penalizes large errors more (sensitivity to outliers)
- **MAPE**: Percentage error (scale-independent)

**Clinical metrics**:
- **Early warning time**: How many hours ahead can we predict deterioration?
- **False alarm rate**: <5% for clinical acceptance
- **Missed event rate**: <1% for critical events (cardiac arrest)

### 7.4 Computational Cost

**Inference time** (forecast 12 hours ahead, 5-min intervals = 144 steps):
- **ARIMA**: ~10 ms (CPU)
- **Holt-Winters**: ~5 ms (CPU)
- **LSTM**: ~50 ms (GPU), ~500 ms (CPU)
- **Transformer**: ~100 ms (GPU), ~2 s (CPU)

**Memory footprint**:
- **ARIMA**: <1 MB (model parameters)
- **LSTM**: 10-100 MB (depends on architecture)
- **Transformer**: 100 MB - 1 GB

## 8. Key Takeaways

### What We Learned

1. **Time series forecasting predicts future vital signs**
   - Critical for early warning systems (sepsis, cardiac arrest)
   - Enables proactive interventions (6+ hours ahead)
   - Requires careful preprocessing (stationarity, missing data)

2. **ARIMA is the gold standard for univariate forecasting**
   - ARIMA(p, d, q): AR + differencing + MA
   - Built-in uncertainty quantification (prediction intervals)
   - Works with small datasets (100-1,000 samples)
   - Fast inference (<10 ms)

3. **Exponential smoothing captures trend and seasonality**
   - Holt-Winters: Level + trend + seasonality
   - Simpler than ARIMA (fewer parameters)
   - Good for circadian patterns in vital signs

4. **Uncertainty grows with forecast horizon**
   - Short-term (0.5-1h): High confidence
   - Long-term (4-6h): Lower confidence
   - Clinical decisions must account for uncertainty

5. **Classical vs deep learning trade-offs**
   - Classical (ARIMA): Small data, interpretable, fast
   - Deep learning (LSTM): Large data, multivariate, complex patterns
   - Choose based on data availability and clinical requirements

### Connections to Book Chapters

- **Journey 7.3**: Sepsis Prediction (forecasting vital sign deterioration)
- **Notebook 7.1**: Signal Preprocessing (clean data improves forecasts)
- **Notebook 7.2**: Feature Extraction (engineered features complement forecasts)
- **Chapter 5**: Evaluation (forecast metrics, confidence intervals)

### Next Steps

In the next notebooks, we will:
- **Notebook 7.6**: Real-time processing pipelines for ICU monitoring
- **Journey 7.3**: Complete sepsis prediction with multivariate time series
- **Journey 7.4**: Compare LSTM-based forecasting to ARIMA

---

## Exercises

1. **Tune ARIMA parameters**:
   - Try different (p, d, q) values: (1,1,1), (3,1,0), (2,1,2)
   - Use AIC/BIC for model selection
   - Which order gives the best MAE?

2. **Add seasonality**:
   - Generate 7-day time series with circadian rhythm
   - Fit SARIMA (Seasonal ARIMA) with period=24h
   - Compare to non-seasonal ARIMA

3. **Multivariate forecasting**:
   - Generate heart rate + blood pressure time series
   - Implement VARIMA (Vector ARIMA)
   - Does multivariate improve accuracy?

4. **Robust forecasting with outliers**:
   - Add outliers (artifacts) to test data
   - Implement rolling window ARIMA (retrain every hour)
   - Compare static vs adaptive models

5. **Clinical decision thresholds**:
   - Set heart rate alarm at 120 BPM (tachycardia)
   - Calculate: How many hours ahead can ARIMA predict alarm?
   - Compute false alarm rate from confidence intervals

6. **Real-world data**:
   - Download MIMIC-III ICU database (https://physionet.org/content/mimiciii/)
   - Extract heart rate time series for sepsis patients
   - Forecast 6 hours ahead and evaluate performance

7. **LSTM implementation** (advanced):
   - Install TensorFlow/PyTorch
   - Implement LSTM forecasting model
   - Compare LSTM vs ARIMA on same dataset
   - Which performs better? Which is more interpretable?

---

*This notebook is part of "AI in Healthcare" (Volume 1)*  
*Full implementation available in the complete book companion code*